In [1]:
import pandas as pd
import numpy as np

np.random.seed(42)
n = 200

df = pd.DataFrame({
    'customer_id': range(1, n+1),
    'age': np.random.randint(18, 70, n).astype(float),
    'gender': np.random.choice(['Male', 'Female', None], n, p=[0.45, 0.45, 0.1]),
    'tenure_months': np.random.randint(1, 72, n),
    'monthly_charge': np.random.normal(70, 20, n).round(2),
    'contract_type': np.random.choice(['Month-to-month', '1 year', '2 year'], n),
    'total_charge': np.random.normal(2000, 800, n).round(2),
    'churn': np.random.choice(['Yes', 'No'], n, p=[0.3, 0.7])
})

# 결측치 인위 삽입
df.loc[np.random.choice(n, 15, replace=False), 'age'] = np.nan
df.loc[np.random.choice(n, 10, replace=False), 'total_charge'] = np.nan

# 이상치 삽입
df.loc[np.random.choice(n, 5, replace=False), 'monthly_charge'] = np.random.uniform(500, 1000, 5)

In [2]:
### age의 결측치는 중앙값으로, gender의 결측치는 최빈값으로 대체하세요.

In [3]:
df['age'] = df['age'].fillna(df['age'].median())

In [5]:
df['gender'] = df['gender'].fillna(df['gender'].mode()[0])

In [7]:
### total_charge의 결측치는 tenure_months * monthly_charge로 추정하여 채우세요.

In [8]:
df['total_charge'] = df['total_charge'].fillna(df['tenure_months'] * df['monthly_charge'])

In [10]:
### monthly_charge의 이상치를 IQR 방법으로 탐지하고, 상한을 초과하는 값은 상한값으로 대체(capping)하세요

In [15]:
q1 = df['monthly_charge'].quantile(0.25)
q3 = df['monthly_charge'].quantile(0.75)
iqr = q3 - q1

In [17]:
upper_bound = q3 + (1.5 * iqr)
lower_bound = q1 - (1.5 * iqr)

In [18]:
df.loc[df['monthly_charge'] > upper_bound, 'monthly_charge'] = upper_bound

### 원-핫 인코딩(빈칸채우기형)
대상 데이터프레임: df
gender, contract_type을 원-핫 인코딩하고 결과를 df_enc에 저장하세요
아래 코드 빈칸을 채우세요

In [22]:
df_enc = pd.get_dummies(data = df, columns = ['gender', 'contract_type'])

### 라벨 인코딩(오류정정형)
churn을 0/1로 라벨 인코딩(Yes=1, No=0)하려는데 에러가 납니다. 고치세요.

In [24]:
df_enc

,customer_id,age,tenure_months,monthly_charge,total_charge,churn,gender_Female,gender_Male,contract_type_1 year,contract_type_2 year,contract_type_Month-to-month
0,1,56.0,67,72.84,4880.28,Yes,False,True,False,True,False
1,2,69.0,26,90.74,2352.77,No,True,False,False,False,True
2,3,43.0,16,63.81,2999.84,Yes,True,False,True,False,False
3,4,32.0,51,46.60,2336.26,No,True,False,False,False,True
4,5,60.0,57,67.84,1779.35,Yes,False,True,False,False,True
...,...,...,...,...,...,...,...,...,...,...,...
195,196,43.0,46,54.70,1405.99,No,True,False,False,True,False
196,197,30.0,35,88.32,1257.38,No,True,False,False,False,True
197,198,58.0,6,91.81,2475.50,No,True,False,False,True,False
198,199,20.0,69,104.12,2832.16,No,False,True,False,True,False


In [25]:
from sklearn.preprocessing import LabelEncoder

In [26]:
le = LabelEncoder()

In [27]:
df_enc['churn'] = le.fit_transform(df_enc['churn'])

### 8. 스케일링(일반형)
age, tenure_months, monthly_charge, total_charge를 StandardScaler로 스케일링하세요
스케일러는 ss 변수에 저장

In [29]:
from sklearn.preprocessing import StandardScaler
ss = StandardScaler()

In [30]:
cols = ['age', 'tenure_months', 'monthly_charge', 'total_charge']

In [31]:
df_enc[cols] = ss.fit_transform(df_enc[cols])

### train/test 분리(일반형)
customer_id 제외, churn을 y로, 나머지를 X로
8:2 분리, random_state=42, 변수명: X_train, X_valid, y_train, y_valid

In [33]:
df_enc.drop(columns = ['customer_id'])

,age,tenure_months,monthly_charge,total_charge,churn,gender_Female,gender_Male,contract_type_1 year,contract_type_2 year,contract_type_Month-to-month
0,0.877484,1.610815,-0.010636,2.891111,1,False,True,False,True,False
1,1.784264,-0.448537,0.784133,0.252746,0,True,False,False,False,True
2,-0.029296,-0.950818,-0.411572,0.928196,1,True,False,True,False,False
3,-0.796571,0.807166,-1.175705,0.235511,0,True,False,False,False,True
4,1.156493,1.108534,-0.232638,-0.345824,1,False,True,False,False,True
...,...,...,...,...,...,...,...,...,...,...
195,-0.029296,0.556025,-0.816061,-0.735560,0,True,False,False,True,False
196,-0.936076,0.003516,0.676684,-0.890688,0,True,False,False,False,True
197,1.016989,-1.453099,0.831642,0.380858,0,True,False,False,True,False
198,-1.633599,1.711272,1.378212,0.753161,0,False,True,False,True,False


In [34]:
y = df_enc['churn']
X = df_enc.drop('churn', axis = 1)

In [38]:
from sklearn.model_selection import train_test_split
X_train, y_train, X_test, y_test = train_test_split(X, y, test_size = 0.2)